In [60]:

import os
import sys
import pandas as pd
import numpy as np
import kagglehub

In [61]:
# Load dataset from Kaggle
path = kagglehub.dataset_download("sayeeduddin/netflix-2025user-behavior-dataset-210k-records")

print(f" Dataset path:{path}\n")
# Dictionary to hold dataframes

file_map = {
    "users" : "users.csv",
    "movies": "movies.csv",
    "watch": "watch_history.csv",
    "recs": "recommendation_logs.csv",
    "search": "search_logs.csv",
    "reviews": "reviews.csv"
}

dataframes = {}

for key,filename in file_map.items():
    file_path = os.path.join(path,filename)

    if not os.path.exists(file_path):
        print(f"Error: File {filename} not found in path {path}")
        continue
        
    dataframes[key] = pd.read_csv(file_path)
    print(f"Loaded {key}: {dataframes[key].shape}")

# dataframes["users"].head()


 Dataset path:/Users/urvashijha/.cache/kagglehub/datasets/sayeeduddin/netflix-2025user-behavior-dataset-210k-records/versions/1

Loaded users: (10300, 16)
Loaded movies: (1040, 18)
Loaded watch: (105000, 12)
Loaded recs: (52000, 11)
Loaded search: (26500, 11)
Loaded reviews: (15450, 12)


In [62]:
# check the dataframes
for key, df in dataframes.items():
    print(f"\n--- {key.upper()} ---")
    print(df.head())
    print(df.info())    


--- USERS ---
      user_id                      email first_name last_name   age  gender  \
0  user_00001   figueroajohn@example.org      Erica     Garza  43.0    Male   
1  user_00002      blakeerik@example.com     Joshua   Bernard  38.0    Male   
2  user_00003        smiller@example.net    Barbara  Williams  32.0  Female   
3  user_00004  mitchellclark@example.com    Chelsea  Ferguson  11.0    Male   
4  user_00005      richard13@example.net      Jason    Foster  21.0  Female   

  country state_province                city subscription_plan  \
0     USA  Massachusetts  North Jefferyhaven             Basic   
1     USA          Texas      North Noahstad          Premium+   
2     USA       Michigan          Traciebury          Standard   
3     USA           Ohio          South Noah          Standard   
4     USA        Arizona         West Donald          Standard   

  subscription_start_date  is_active  monthly_spend primary_device  \
0              2024-04-08       True       

In [63]:
# Configuration
dat_cols = {
    "users": ["subscription_start_date","created_at"],
    "watch": ["watch_date"],
    "recs": ["recommendation_date"],
    "search": ["search_date"],
    "reviews": ["review_date"]
}
#columns that must be positive
pos_cols = ["age","household_size"] # must be > 0
non_neg_cols = ["monthly_spend"] # must be >= 0


#standardize column names

for key in dataframes:
    dataframes[key].columns = (
        dataframes[key].columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
    )
# Convert date columns to datetime
for table,cols in dat_cols.items():
    for col in cols:
        if col in dataframes[table].columns:
            dataframes[table][col] = pd.to_datetime(dataframes[table][col], errors="coerce")


In [64]:
# remove duplicates
for key in dataframes:
    before = dataframes[key].shape[0]
    dataframes[key] = dataframes[key].drop_duplicates()
    dropped = before - len(dataframes[key])
    print(f"{key}: Dropped {dropped} duplicate rows")   
print()

users: Dropped 300 duplicate rows
movies: Dropped 40 duplicate rows
watch: Dropped 5000 duplicate rows
recs: Dropped 2000 duplicate rows
search: Dropped 1500 duplicate rows
reviews: Dropped 450 duplicate rows



In [65]:
# remove impossible values
users = dataframes["users"]
for col in pos_cols:
    if col in users.columns:
        before = users.shape[0]
        users = users[users[col] > 0]
        dropped = before - users.shape[0]
        print(f"users: Dropped {dropped} rows with non-positive {col}")
for col in non_neg_cols:
    if col in users.columns:
        before = users.shape[0]
        users = users[users[col] >= 0]
        dropped = before - users.shape[0]
        print(f"users: Dropped {dropped} rows with negative {col}")
dataframes["users"] = users


users: Dropped 1216 rows with non-positive age
users: Dropped 1319 rows with non-positive household_size
users: Dropped 730 rows with negative monthly_spend


In [66]:
# identify missing values
for key, df in dataframes.items():
    missing = df.isnull().sum()
    print(f"\n--- {key.upper()} Missing Values ---")
    print(missing[missing > 0]) 

    if missing.empty:
        print("No missing values")

    total_missing = missing.sum()
    print(f"Total missing values in {key}: {total_missing}")

    for col, count in missing[missing > 0].items():
        pct = (count / len(df)) * 100
        dtype = str(df[col].dtype)

        if pct > 50:
            action = "Consider dropping"
        elif dtype == "float64" or dtype == "int64":
            action = "Fill with median"
        elif "datetime64[ns]" in dtype:
            action = "investigate"
        else:
            action = "Fill with mode or 'Unknown'"
        print(f"{col}: {count} missing ({pct:.2f}%) - {action}")



--- USERS Missing Values ---
gender    554
dtype: int64
Total missing values in users: 554
gender: 554 missing (8.23%) - Fill with mode or 'Unknown'

--- MOVIES Missing Values ---
genre_secondary       643
imdb_rating           144
production_budget     647
box_office_revenue    678
number_of_seasons     725
number_of_episodes    695
dtype: int64
Total missing values in movies: 3532
genre_secondary: 643 missing (64.30%) - Consider dropping
imdb_rating: 144 missing (14.40%) - Fill with median
production_budget: 647 missing (64.70%) - Consider dropping
box_office_revenue: 678 missing (67.80%) - Consider dropping
number_of_seasons: 725 missing (72.50%) - Consider dropping
number_of_episodes: 695 missing (69.50%) - Consider dropping

--- WATCH Missing Values ---
watch_duration_minutes    11771
progress_percentage        8094
user_rating               79886
dtype: int64
Total missing values in watch: 99751
watch_duration_minutes: 11771 missing (11.77%) - Fill with median
progress_percentag

In [67]:
# Impute missing values based on the above analysis

# missing imdb ratings skipped at this stage


# ── 5a. USERS ────────────────────────────────────────────────────────────────

if "gender" in dataframes["users"].columns:
    n = dataframes["users"]["gender"].isna().sum()
    dataframes["users"]["gender"] = dataframes["users"]["gender"].fillna("Unknown")
    print(f"  users.gender: filled {n:,} nulls with 'Unknown'")

# ── 5b. MOVIES ───────────────────────────────────────────────────────────────

movies = dataframes["movies"]

# genre_secondary → "None" (no secondary genre exists, not unknown)
if "genre_secondary" in movies.columns:
    n = movies["genre_secondary"].isna().sum()
    movies["genre_secondary"] = movies["genre_secondary"].fillna("None")
    print(f"  movies.genre_secondary: filled {n:,} nulls with 'None'")

# number_of_seasons + number_of_episodes → create is_series flag then drop
if "number_of_seasons" in movies.columns or "number_of_episodes" in movies.columns:
    seasons  = movies["number_of_seasons"]  if "number_of_seasons"  in movies.columns else pd.Series(False, index=movies.index)
    episodes = movies["number_of_episodes"] if "number_of_episodes" in movies.columns else pd.Series(False, index=movies.index)
    movies["is_series"] = (seasons.notna() | episodes.notna()).astype(int)
    print(f"  movies: created is_series flag ({movies['is_series'].sum():,} series, {(movies['is_series']==0).sum():,} films)")
elif "is_series" in movies.columns:
    print(f"  movies: is_series already exists — skipping")

# drop high-null columns irrelevant to user churn
cols_to_drop = ["production_budget", "box_office_revenue",
                "number_of_seasons", "number_of_episodes"]
cols_to_drop = [c for c in cols_to_drop if c in movies.columns]
if cols_to_drop:
    movies = movies.drop(columns=cols_to_drop)
    print(f"  movies: dropped {cols_to_drop}")
dataframes["movies"] = movies


# ── 5c. WATCH ────────────────────────────────────────────────────────────────

watch = dataframes["watch"]

# user_rating (79.9%) → create has_rated flag then drop
if "user_rating" in watch.columns:
    watch["has_rated"] = watch["user_rating"].notna().astype(int)
    rated_pct = watch["has_rated"].mean() * 100
    print(f"  watch: created has_rated flag ({rated_pct:.1f}% of watches rated)")
    watch = watch.drop(columns=["user_rating"])
elif "has_rated" in watch.columns:
    print(f"  watch: has_rated already exists — skipping")

# watch_duration_minutes + progress_percentage → fill with median
for col in ["watch_duration_minutes", "progress_percentage"]:
    if col in watch.columns:
        median_val = watch[col].median()
        n = watch[col].isna().sum()
        watch[col] = watch[col].fillna(median_val)
        print(f"  watch.{col}: filled {n:,} nulls with median ({median_val:.2f})")

dataframes["watch"] = watch

# ── 5d. RECS ─────────────────────────────────────────────────────────────────

# recommendation_score → fill with median
if "recommendation_score" in dataframes["recs"].columns:
    median_val = dataframes["recs"]["recommendation_score"].median()
    n = dataframes["recs"]["recommendation_score"].isna().sum()
    dataframes["recs"]["recommendation_score"] = dataframes["recs"]["recommendation_score"].fillna(median_val)
    print(f"  recs.recommendation_score: filled {n:,} nulls with median ({median_val:.3f})")

# algorithm_version → fill with mode
if "algorithm_version" in dataframes["recs"].columns:
    mode_val = dataframes["recs"]["algorithm_version"].mode()[0]
    n = dataframes["recs"]["algorithm_version"].isna().sum()
    dataframes["recs"]["algorithm_version"] = dataframes["recs"]["algorithm_version"].fillna(mode_val)
    print(f"  recs.algorithm_version: filled {n:,} nulls with mode ({mode_val})")

# ── 5e. SEARCH ───────────────────────────────────────────────────────────────

search = dataframes["search"]

# clicked_result_position (51%) → create clicked flag then drop
if "clicked_result_position" in search.columns:
    search["clicked"] = search["clicked_result_position"].notna().astype(int)
    clicked_pct = search["clicked"].mean() * 100
    print(f"  search: created clicked flag ({clicked_pct:.1f}% of searches had a click)")
    search = search.drop(columns=["clicked_result_position"])
elif "clicked" in search.columns:
    print(f"  search: clicked already exists — skipping")

# search_duration_seconds → fill with median
if "search_duration_seconds" in search.columns:
    median_val = search["search_duration_seconds"].median()
    n = search["search_duration_seconds"].isna().sum()
    search["search_duration_seconds"] = search["search_duration_seconds"].fillna(median_val)
    print(f"  search.search_duration_seconds: filled {n:,} nulls with median ({median_val:.1f})")

dataframes["search"] = search

# ── 5f. REVIEWS ──────────────────────────────────────────────────────────────

# helpful_votes + total_votes → fill with 0 (missing = nobody voted yet)
for col in ["helpful_votes", "total_votes"]:
    if col in dataframes["reviews"].columns:
        n = dataframes["reviews"][col].isna().sum()
        dataframes["reviews"][col] = dataframes["reviews"][col].fillna(0)
        print(f"  reviews.{col}: filled {n:,} nulls with 0")

# review_text → fill with empty string (user rated but wrote nothing)
if "review_text" in dataframes["reviews"].columns:
    n = dataframes["reviews"]["review_text"].isna().sum()
    dataframes["reviews"]["review_text"] = dataframes["reviews"]["review_text"].fillna("")
    print(f"  reviews.review_text: filled {n:,} nulls with empty string")

# sentiment_score → fill with median
if "sentiment_score" in dataframes["reviews"].columns:
    median_val = dataframes["reviews"]["sentiment_score"].median()
    n = dataframes["reviews"]["sentiment_score"].isna().sum()
    dataframes["reviews"]["sentiment_score"] = dataframes["reviews"]["sentiment_score"].fillna(median_val)
    print(f"  reviews.sentiment_score: filled {n:,} nulls with median ({median_val:.3f})")

# ── 5g. Catch-all — any remaining object nulls across all tables ──────────────
caught_any = False

for key, df in dataframes.items():
    for col in df.select_dtypes("float64").columns:
       if col == "imdb_rating":
            continue                          # skip — handled in Step 5.5
       n = df[col].isna().sum()
       if n > 0:
            df[col] = df[col].fillna(df[col].median())
            print(f"  ⚠️  {key}.{col}: catch-all filled {n:,} nulls with median")
            caught_any = True
    for col in df.select_dtypes("object").columns:
        n = df[col].isna().sum()
        if n > 0:
            df[col] = df[col].fillna("Unknown")
            print(f"  ⚠️  {key}.{col}: catch-all filled {n:,} nulls with Unknown")
            caught_any = True

if not caught_any:
   print("  ✅ No nulls caught — all columns explicitly handled")

print()




  users.gender: filled 554 nulls with 'Unknown'
  movies.genre_secondary: filled 643 nulls with 'None'
  movies: created is_series flag (305 series, 695 films)
  movies: dropped ['production_budget', 'box_office_revenue', 'number_of_seasons', 'number_of_episodes']
  watch: created has_rated flag (20.1% of watches rated)
  watch.watch_duration_minutes: filled 11,771 nulls with median (51.20)
  watch.progress_percentage: filled 8,094 nulls with median (49.80)
  recs.recommendation_score: filled 5,005 nulls with median (0.553)
  recs.algorithm_version: filled 2,522 nulls with mode (v1.4)
  search: created clicked flag (48.9% of searches had a click)
  search.search_duration_seconds: filled 1,215 nulls with median (16.8)
  reviews.helpful_votes: filled 1,757 nulls with 0
  reviews.total_votes: filled 1,757 nulls with 0
  reviews.review_text: filled 767 nulls with empty string
  reviews.sentiment_score: filled 1,177 nulls with median (0.669)
  ✅ No nulls caught — all columns explicitly hand

In [68]:
# Fill imdb_rating via OMDB API
# Get your free API key at: http://www.omdbapi.com/apikey.aspx

import requests
import time

OMDB_API_KEY = os.getenv("OMDB_API_KEY")  #

print("Step 5.5: Filling imdb_rating via OMDB API...")
print("-" * 50)

movies           = dataframes["movies"]
missing_mask     = movies["imdb_rating"].isna()
n_missing_before = missing_mask.sum()

print(f"  imdb_rating nulls before API : {n_missing_before}\n")

fetched   = 0
not_found = 0
errors    = 0

for idx, row in movies[missing_mask].iterrows():
    title = row["title"]
    try:
        response = requests.get(
            "http://www.omdbapi.com/",
            params={"t": title, "apikey": OMDB_API_KEY},
            timeout=5
        )
        data = response.json()

        if data.get("Response") == "True" and data.get("imdbRating") != "N/A":
            movies.at[idx, "imdb_rating"] = float(data["imdbRating"])
            print(f"  ✅ {title:<40} → {data['imdbRating']}")
            fetched += 1
        else:
            print(f"  ⚠️  {title:<40} → not found")
            not_found += 1

    except Exception as e:
        print(f"  ❌ {title:<40} → error: {e}")
        errors += 1

    time.sleep(0.1)

dataframes["movies"] = movies

# ── Summary ───────────────────────────────────────────────────────────────────

n_missing_after = dataframes["movies"]["imdb_rating"].isna().sum()

print("\n" + "=" * 50)
print("  OMDB FETCH SUMMARY")
print("=" * 50)
print(f"  Nulls before API call  : {n_missing_before}")
print(f"  Filled by OMDB         : {fetched}")
print(f"  Not found in OMDB      : {not_found}")
print(f"  Errors                 : {errors}")
print(f"  Nulls remaining        : {n_missing_after}")
print("=" * 50)

if n_missing_after > 0:
    print(f"\n  ⚠️  {n_missing_after} nulls still remaining — decide on imputation strategy:")
    print(f"      Option 1 → fill with median ({movies['imdb_rating'].median():.2f})")
    print(f"      Option 2 → fill with mean   ({movies['imdb_rating'].mean():.2f})")
    print(f"      Option 3 → drop these rows")
    print(f"      Option 4 → try TMDB API for remaining titles")
    # show exactly which titles are still missing so you can investigate
    still_missing_titles = dataframes["movies"][dataframes["movies"]["imdb_rating"].isna()]["title"].tolist()
    print(f"\n  Titles still missing:")
    for t in still_missing_titles:
        print(f"    - {t}")
else:
    print("\n  ✅ All imdb_rating nulls resolved by OMDB")

print("\nStep 5.5 complete ✅")

Step 5.5: Filling imdb_rating via OMDB API...
--------------------------------------------------
  imdb_rating nulls before API : 144

  ⚠️  Dragon Legend                            → not found
  ⚠️  Battle Story                             → not found
  ⚠️  Day Dream                                → not found
  ⚠️  Hero Empire                              → not found
  ⚠️  First Mystery                            → not found
  ⚠️  Legend Hero                              → not found
  ⚠️  House Hero                               → not found
  ⚠️  Empire King                              → not found
  ⚠️  Kingdom War                              → not found
  ⚠️  Secret Legend                            → not found
  ⚠️  Day Queen                                → not found
  ⚠️  Storm Warrior                            → not found
  ⚠️  Dark Ice                                 → not found
  ⚠️  Little War                               → not found
  ⚠️  Mystery Mystery                  

In [69]:
# Median imputation for any remaining nulls after API fill
# Fills remaining nulls with genre-based median (primary strategy)
# Falls back to global median if a genre has no rated movies at all


movies         = dataframes["movies"]
missing_mask   = movies["imdb_rating"].isna()
n_before       = missing_mask.sum()

print(f"  Nulls before imputation : {n_before}")

if n_before == 0:
    print("  ✅ No nulls remaining — skipping")

else:
    global_median = movies["imdb_rating"].median()
    print(f"  Global median           : {global_median:.2f}")
    print()

    genre_filled  = 0
    global_filled = 0

    for idx, row in movies[missing_mask].iterrows():
        genre = row["genre_primary"]

        # get median rating for this specific genre (excluding nulls)
        genre_ratings = movies[
            (movies["genre_primary"] == genre) &
            (movies["imdb_rating"].notna())
        ]["imdb_rating"]

        if len(genre_ratings) > 0:
            genre_median = genre_ratings.median()
            movies.at[idx, "imdb_rating"] = genre_median
            print(f"  ✅ '{row['title']:<35} [{genre}] → genre median ({genre_median:.2f})")
            genre_filled += 1
        else:
            # fallback — genre has no rated movies at all
            movies.at[idx, "imdb_rating"] = global_median
            print(f"  ⚠️  '{row['title']:<35} [{genre}] → global median ({global_median:.2f}) [no genre data]")
            global_filled += 1

    dataframes["movies"] = movies

    # ── Summary ───────────────────────────────────────────────────────────────

    n_after = dataframes["movies"]["imdb_rating"].isna().sum()

    print("\n" + "=" * 50)
    print("  FINAL IMPUTATION SUMMARY")
    print("=" * 50)
    print(f"  Nulls before            : {n_before}")
    print(f"  Filled by genre median  : {genre_filled}")
    print(f"  Filled by global median : {global_filled}")
    print(f"  Nulls remaining         : {n_after}")
    print("=" * 50)

    if n_after == 0:
        print("\n  ✅ All imdb_rating nulls resolved")
    else:
        print(f"\n  ❌ {n_after} nulls still remain — investigate")





  Nulls before imputation : 144
  Global median           : 6.40

  ✅ 'Dragon Legend                       [History] → genre median (6.35)
  ✅ 'Battle Story                        [Sci-Fi] → genre median (6.70)
  ✅ 'Day Dream                           [Western] → genre median (6.30)
  ✅ 'Hero Empire                         [Sci-Fi] → genre median (6.70)
  ✅ 'First Mystery                       [Sci-Fi] → genre median (6.70)
  ✅ 'Legend Hero                         [Biography] → genre median (6.50)
  ✅ 'House Hero                          [Crime] → genre median (6.50)
  ✅ 'Empire King                         [Comedy] → genre median (6.30)
  ✅ 'Kingdom War                         [Biography] → genre median (6.50)
  ✅ 'Secret Legend                       [Family] → genre median (6.25)
  ✅ 'Day Queen                           [Sport] → genre median (6.40)
  ✅ 'Storm Warrior                       [Sci-Fi] → genre median (6.70)
  ✅ 'Dark Ice                            [Comedy] → genre median

In [70]:
# cross table integrity: remove rows in child tables whose IDs don't exist in parent tables
# Remove rows in child tables whose IDs don't exist in parent tables.
# ghost user_id or movie_id = data we can't link to anyone = useless for modeling.

print("Step 6: Cross-table integrity check...")

valid_users  = set(dataframes["users"]["user_id"])
valid_movies = set(dataframes["movies"]["movie_id"])

checks = {
    "watch":   [("user_id", valid_users), ("movie_id", valid_movies)],
    "recs":    [("user_id", valid_users), ("movie_id", valid_movies)],
    "reviews": [("user_id", valid_users), ("movie_id", valid_movies)],
    "search":  [("user_id", valid_users)],   # search has no movie_id
}

for table, id_checks in checks.items():
    for col, valid_ids in id_checks:         # no .items() — iterating list of tuples
        if col in dataframes[table].columns:
            before = len(dataframes[table])
            dataframes[table] = dataframes[table][
                dataframes[table][col].isin(valid_ids)
            ]
            removed = before - len(dataframes[table])
            print(f"  {table}.{col}: removed {removed:,} ghost IDs")
        else:
            print(f"  ⚠️  {table}.{col} not found — skipping")

print()



Step 6: Cross-table integrity check...
  watch.user_id: removed 32,526 ghost IDs
  watch.movie_id: removed 0 ghost IDs
  recs.user_id: removed 16,399 ghost IDs
  recs.movie_id: removed 0 ghost IDs
  reviews.user_id: removed 4,869 ghost IDs
  reviews.movie_id: removed 0 ghost IDs
  search.user_id: removed 8,207 ghost IDs



In [71]:

print("=" * 45)
print("PREPROCESSING COMPLETE — Final shapes:")
print("=" * 45)
for key, df in dataframes.items():
    nulls = df.isna().sum().sum()
    print(f"  {key:<10} {df.shape[0]:>8,} rows × {df.shape[1]:>2} cols "
          f"| {nulls:,} nulls remaining")


PREPROCESSING COMPLETE — Final shapes:
  users         6,735 rows × 16 cols | 0 nulls remaining
  movies        1,000 rows × 15 cols | 0 nulls remaining
  watch        67,474 rows × 12 cols | 0 nulls remaining
  recs         33,601 rows × 11 cols | 0 nulls remaining
  search       16,793 rows × 11 cols | 0 nulls remaining
  reviews      10,131 rows × 12 cols | 0 nulls remaining


In [ ]:
# Save cleaned dataframes to new CSV and pickle files for modeling

save_path = "cleaned_data"
os.makedirs(save_path, exist_ok=True)

for key, df in dataframes.items():
    # CSV — human readable, shareable anywhere
    df.to_csv(os.path.join(save_path, f"{key}_cleaned.csv"), index=False)
    # Pickle — preserves dtypes for direct use in modeling
    df.to_pickle(os.path.join(save_path, f"{key}_cleaned.pkl"))
    print(f"  ✅ {key:<10} saved as CSV + pickle")

print(f"\nFiles saved to '{save_path}/'")
print("Share the folder with your collaborator.")
print("Tell them to use .pkl files to preserve datetime dtypes.")